# 02 — Perception API Wrapper v2

Improvements over v1:
- `session_id` — groups queries by caller (attacker vs normal user)
- `delta_ms` — time since last query in session (burst detection)
- `phash` — perceptual hash for image similarity (not just exact match)

**Log format:** `data/query_log_v2.jsonl`

In [ ]:
import torch
import subprocess
import sys

print("=== Python ===")
print(f"  {sys.version}")

print("\n=== PyTorch ===")
print(f"  Version : {torch.__version__}")
print(f"  CUDA available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  GPU count : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1e9
        print(f"  GPU {i}     : {props.name} ({mem_gb:.1f} GB VRAM)")
    print(f"  Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  ⚠️  No CUDA GPU detected — YOLO will run on CPU (slower)")

print("\n=== nvidia-smi ===")
try:
    smi = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,utilization.gpu","--format=csv,noheader"], text=True)
    for line in smi.strip().split("\n"):
        print(f"  {line}")
except FileNotFoundError:
    print("  nvidia-smi not found")

print("\n=== Packages ===")
import ultralytics, cv2, imagehash
print(f"  ultralytics : {ultralytics.__version__}")
print(f"  opencv      : {cv2.__version__}")
print(f"  imagehash   : {imagehash.__version__}")

print("\n=== YOLO device test ===")
from ultralytics import YOLO as _YOLO
import numpy as np
_m = _YOLO("../model/yolo11n.pt")
_dummy = np.zeros((640, 640, 3), dtype=np.uint8)
_r = _m(_dummy, verbose=False)
_device = str(_m.device)
print(f"  YOLO running on: {_device}")
if "cuda" in _device or "0" in _device:
    print("  ✅ GPU confirmed")
else:
    print("  ⚠️  Running on CPU")
del _m, _dummy, _r

## 1. Setup

In [ ]:
# pip install imagehash Pillow ultralytics opencv-python
import hashlib
import json
import time
from pathlib import Path

import cv2
import numpy as np
import imagehash
from PIL import Image
from ultralytics import YOLO

LOG_FILE = Path("../logs/query_log_v2.jsonl")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

model = YOLO("../models/yolo11n.pt")

# Per-session state: tracks last query time per session
_session_last_ts: dict = {}
_global_query_id: int = 0

print(f"Model loaded. Log → {LOG_FILE.resolve()}")

## 2. Core API Function

In [ ]:
def _to_pil(image):
    """Convert image input to PIL for pHash."""
    if isinstance(image, (str, Path)):
        return Image.open(image).convert("RGB")
    else:
        # numpy BGR (cv2) → RGB PIL
        return Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))


def query(image, session_id="default", source_label="unknown"):
    """
    Simulated greybox perception API.

    Args:
        image:        file path (str/Path) OR numpy array (BGR)
        session_id:   caller identity string — group per attacker/user
        source_label: semantic tag for log ('normal_video', 'attack_knockoff',
                      'attack_zoo', 'attack_hsja', 'attack_meaod')

    Returns:
        list of dicts: [{class, class_name, conf, bbox}, ...]
    """
    global _global_query_id
    _global_query_id += 1

    now = time.time()

    # --- Inter-query delta (per session) ---
    last_ts = _session_last_ts.get(session_id)
    delta_ms = round((now - last_ts) * 1000, 1) if last_ts else None
    _session_last_ts[session_id] = now

    # --- Hashes ---
    pil_img = _to_pil(image)
    phash = str(imagehash.phash(pil_img))          # perceptual — similarity-aware

    if isinstance(image, (str, Path)):
        md5 = hashlib.md5(Path(image).read_bytes()).hexdigest()
    else:
        md5 = hashlib.md5(image.tobytes()).hexdigest()

    # --- Inference ---
    t0 = time.perf_counter()
    results = model(image, verbose=False)
    latency_ms = round((time.perf_counter() - t0) * 1000, 1)

    r = results[0]

    # --- Response (what attacker receives) ---
    detections = []
    for box in r.boxes:
        detections.append({
            "class":      int(box.cls[0]),
            "class_name": model.names[int(box.cls[0])],
            "conf":       round(float(box.conf[0]), 4),
            "bbox":       [round(x, 2) for x in box.xyxy[0].tolist()]
        })

    # --- Log entry (what monitor reads) ---
    entry = {
        "query_id":     _global_query_id,
        "session_id":   session_id,
        "timestamp":    now,
        "delta_ms":     delta_ms,          # None for first query in session
        "source":       source_label,
        "md5":          md5,               # exact identity
        "phash":        phash,             # perceptual similarity
        "n_detections": len(detections),
        "latency_ms":   latency_ms,
        "detections":   detections
    }

    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

    return detections

## 3. Mode A — Single Image

In [ ]:
IMAGE_PATH = "../data/gettyimage.jpg"

result = query(IMAGE_PATH, session_id="user_normal", source_label="normal_image")

print(f"Detections: {len(result)}")
for d in result:
    print(f"  [{d['class_name']:15s}] conf={d['conf']:.2f}  bbox={d['bbox']}")

## 4. Mode B — Video File (normal baseline traffic)

In [ ]:
VIDEO_PATH  = "../data/dash-cam-video.mp4"
MAX_FRAMES  = None          # None = full video
SAVE_OUTPUT = False
OUTPUT_PATH = "../output/output_annotated_v2.mp4"
SESSION_ID  = "user_normal"

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open: {VIDEO_PATH}")

fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
limit  = min(MAX_FRAMES, total) if MAX_FRAMES else total

print(f"Video: {width}x{height} @ {fps:.1f}fps | {limit} frames | session={SESSION_ID}")

writer = None
if SAVE_OUTPUT:
    writer = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

for i in range(limit):
    ret, frame = cap.read()
    if not ret:
        break

    detections = query(frame, session_id=SESSION_ID, source_label="normal_video")

    if writer:
        for d in detections:
            x1, y1, x2, y2 = [int(v) for v in d["bbox"]]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"{d['class_name']} {d['conf']:.2f}",
                        (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        writer.write(frame)

    if i % 50 == 0:
        print(f"  {i}/{limit}", end="\r")

cap.release()
if writer:
    writer.release()
print(f"\nDone. {i+1} frames logged.")

## 5. Inspect Log + Verify New Fields

In [ ]:
import pandas as pd

records = []
with open(LOG_FILE) as f:
    for line in f:
        e = json.loads(line)
        records.append({
            "query_id":     e["query_id"],
            "session_id":   e["session_id"],
            "source":       e["source"],
            "delta_ms":     e["delta_ms"],
            "phash":        e["phash"],
            "n_detections": e["n_detections"],
        })

df = pd.DataFrame(records)
print(df.tail(10).to_string(index=False))
print(f"\nTotal queries: {len(df)}")
print(f"Sessions: {df['session_id'].unique()}")
print(f"Sources:  {df['source'].unique()}")

## 6. pHash Similarity Check (sanity test)
Demonstrates that pHash captures visual similarity — two consecutive video frames should have low Hamming distance, while unrelated images should have high distance.

In [ ]:
import random

with open(LOG_FILE) as f:
    all_entries = [json.loads(l) for l in f]

# Sample 5 random entries spread across the log
indices = sorted(random.sample(range(len(all_entries)), min(5, len(all_entries))))
samples = [all_entries[i] for i in indices]

print(f"Sampled query_ids: {[s['query_id'] for s in samples]}")
print(f"Sources:           {[s['source'] for s in samples]}")
print()

hashes = [imagehash.hex_to_hash(s["phash"]) for s in samples]

print("pHash pairwise Hamming distances (lower = more similar):")
print(f"{'':>6}", end="")
for i in range(len(hashes)):
    print(f"  [{i}]", end="")
print()
for i, h1 in enumerate(hashes):
    print(f"  [{i}] ", end="")
    for h2 in hashes:
        print(f"  {h1 - h2:3d}", end="")
    print(f"   qid={samples[i]['query_id']}")

print()
print("Expected ranges:")
print("  Consecutive frames : 0-5   (same scene, slight motion)")
print("  Different scenes   : 10-30 (normal video variation)")
print("  ZOO attack         : 0-5   (perturbed same image)")
print("  Knockoff attack    : 25+   (random diverse images)")

In [ ]:
# Measure consecutive frame distances (the missing baseline)
with open(LOG_FILE) as f:
    all_entries = [json.loads(l) for l in f]

# Take 20 consecutive pairs from middle of video
mid = len(all_entries) // 2
pairs = [(all_entries[i], all_entries[i+1]) for i in range(mid, mid+20)]

dists = []
for a, b in pairs:
    h1 = imagehash.hex_to_hash(a["phash"])
    h2 = imagehash.hex_to_hash(b["phash"])
    dists.append(h1 - h2)

print(f"Consecutive frame Hamming distances:")
print(f"  Values : {dists}")
print(f"  Min    : {min(dists)}")
print(f"  Max    : {max(dists)}")
print(f"  Mean   : {sum(dists)/len(dists):.1f}")